# 📊 Exploración óptima de datos a través de Pandas

**Curso de Inteligencia Artificial — Samsung Innovation Campus**
**Capítulo 3: Análisis de datos exploratorios — Unidad 2: Pandas**

En este notebook vamos a recorrer, con código ejecutable y ejercicios, los mismos temas que se ven en la clase:

1. **2.1** Pipelines: estructuras de datos según tipos de datos
2. **2.2** Series de Pandas y DataFrames
3. **2.3** Fusionar y vincular DataFrames (`concat`, `merge`)
4. **2.4** Clasificación de DataFrames e índice múltiple (`sort_values`, `MultiIndex`)
5. **2.5** Estadísticas descriptivas, agrupamiento, *pivoting* y muestreo de datos

> 💡 Cada sección tiene explicación teórica, celdas de código para correr y ejercicios al final con su solución (colapsada) para practicar.


In [ ]:
# Librerías que vamos a usar en todo el notebook
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("pandas:", pd.__version__)
print("numpy :", np.__version__)


pandas: 2.2.2
numpy : 2.0.2


## 2.1. Pipelines: estructuras de datos según tipos de datos

Los datos recopilados pueden clasificarse, según su **perspectiva estructural** (o de esquema), en tres categorías:

| Tipo | Descripción | Ejemplos de formato | Ejemplos de forma en Python |
|---|---|---|---|
| **Estructurado** | Tiene un esquema fijo, se almacena en campos fijos, consistente en valor y formato | Tablas, hojas de cálculo, RDB | `array`, `Matriz`, `DataFrame` |
| **Semiestructurado** | Tiene una estructura formal y metadatos, pero es inconsistente en valor/formato | XML, HTML, JSON, logs | `Diccionario` |
| **No estructurado** | No tiene una estructura de esquema fija | Texto plano, redes sociales, foros | `lista`, `tupla`, `set` |

**Pipeline típico de estructuras de datos en Python:**

```
Fuente (Internet / Archivo / Base de datos)
        │
        ▼
Formato (texto plano, CSV, HTML/XML, JSON, tabla)
        │
        ▼
Tratamiento (lista/tupla/set → array/Matriz → Marco/Serie → Diccionario)
```

La idea central: **sin importar de dónde vengan los datos, para poder analizarlos con Python los terminamos llevando a una estructura común** — en la mayoría de los casos, un `DataFrame` de pandas.

`pandas` está construido **sobre `numpy`**, pero mientras `numpy` trabaja con arrays homogéneos (un solo tipo de dato), `pandas` puede manejar **columnas de distintos tipos** dentro de la misma tabla — igual que una hoja de cálculo o una tabla SQL.


In [ ]:
# Ejemplo: los mismos datos "crudos" en distintas estructuras del pipeline

# 1) No estructurado / semiestructurado: texto plano o lista de diccionarios (como vendría de una API JSON)
datos_json = [
    {"nombre": "Tomás", "edad": 15, "sexo": "Masculino", "escuela": "medio"},
    {"nombre": "Alicia", "edad": 10, "sexo": "Femenino", "escuela": "elemental"},
]

# 2) Estructurado: lo llevamos a un DataFrame de pandas
df_estudiantes = pd.DataFrame(datos_json)
df_estudiantes


,nombre,edad,sexo,escuela
0,Tomás,15,Masculino,medio
1,Alicia,10,Femenino,elemental


## 2.2. Serie de Pandas y DataFrames

Pandas tiene dos estructuras de datos fundamentales:

- **`Series`**: un arreglo unidimensional **etiquetado** (tiene un índice). Es como una columna de una tabla.
- **`DataFrame`**: una tabla bidimensional, formada por **varias `Series` que comparten el mismo índice**. Cada columna puede tener su propio tipo de dato.

### Creación de una `Series`


In [ ]:
# Una Series se puede crear desde una lista, un array de NumPy o un diccionario
serie_edad = pd.Series([23, 22, 24, 23, 20, 21], name="edad")
print(serie_edad)
print()
print("Índice :", serie_edad.index)
print("Valores:", serie_edad.values)
print("dtype  :", serie_edad.dtype)


In [ ]:
# Serie creada desde un diccionario: las claves se vuelven el índice
serie_tipo_sangre = pd.Series({"est_1": "O", "est_2": "A", "est_3": "B", "est_4": "AB"})
serie_tipo_sangre


### Creación de un `DataFrame`

Un `DataFrame` reúne varias `Series` (columnas). Se puede crear a partir de:
- un **diccionario de listas** (cada clave → una columna),
- una **lista de diccionarios** (cada diccionario → una fila),
- un **array de NumPy** (bidimensional) + nombres de columnas,
- o leyendo un **archivo externo** (`pd.read_csv`, `pd.read_excel`, `pd.read_json`, etc.).


In [ ]:
# DataFrame a partir de un diccionario de listas — el ejemplo clásico "Series → DataFrame"
datos = {
    "Sexo":          ["Masculino", "Femenino", "Masculino", "Masculino", "Femenino", "Femenino"],
    "tipo_sangre":   ["3", "2", "4", "3", "1", "2"],
    "Altura":        [23, 22, 24, 23, 20, 21],
    "Peso":          [68.2, 53.0, 80.1, 85.7, 49.5, 52.0],
}

df = pd.DataFrame(datos)
df


,Sexo,tipo_sangre,Altura,Peso
0,Masculino,3,23,68.2
1,Femenino,2,22,53.0
2,Masculino,4,24,80.1
3,Masculino,3,23,85.7
4,Femenino,1,20,49.5
5,Femenino,2,21,52.0


In [ ]:
# DataFrame a partir de un array bidimensional de NumPy (sin nombres de columna al inicio)
arr = np.array([
    ["Tomás", 15, "Masculino", "medio"],
    ["Alicia", 10, "Femenino", "elemental"],
])

df_desde_array = pd.DataFrame(arr)
df_desde_array


In [ ]:
# Como se ve arriba, pandas NO reconoce los nombres de columna automáticamente.
# Hay que asignarlos manualmente, igual que en la clase:
df_desde_array.columns = ["nombre", "edad", "sexo", "escuela"]
df_desde_array


### Lo primero que hay que revisar al cargar un DataFrame

Cada vez que cargamos datos (por ejemplo desde un CSV), conviene inspeccionarlos antes de trabajar con ellos:

| Método / atributo | Para qué sirve |
|---|---|
| `df.head(n)` | Muestra las primeras `n` filas (por defecto 5) |
| `df.tail(n)` | Muestra las últimas `n` filas |
| `df.info()` | Tipo de índice, columnas, valores no nulos, uso de memoria |
| `df.columns` | Nombres de las columnas (es un **atributo**, no un método → sin paréntesis) |
| `df.index` | Índice de filas |
| `df.shape` | Tupla `(filas, columnas)` |
| `df.describe()` | Estadísticas descriptivas de las columnas numéricas |

> ⚠️ Ojo con `df.columns()` — **no es un método**, es un atributo. Si le ponés paréntesis, da error.


In [ ]:
# Vamos a usar un dataset un poco más grande para practicar: el clásico Iris, armado a mano
iris_df = pd.DataFrame({
    "sepal.length": [5.1, 4.9, 4.7, 7.0, 6.4, 6.9, 6.3, 5.8, 7.1, 6.5],
    "sepal.width":  [3.5, 3.0, 3.2, 3.2, 3.2, 3.1, 3.3, 2.7, 3.0, 3.0],
    "petal.length": [1.4, 1.4, 1.3, 4.7, 4.5, 4.9, 6.0, 5.1, 5.9, 5.8],
    "petal.width":  [0.2, 0.2, 0.2, 1.4, 1.5, 1.5, 2.5, 1.9, 2.1, 2.2],
    "variety":      ["Setosa", "Setosa", "Setosa", "Versicolor", "Versicolor",
                      "Versicolor", "Virginica", "Virginica", "Virginica", "Virginica"],
})

iris_df.head()


In [ ]:
iris_df.tail(3)


In [ ]:
iris_df.info()


In [ ]:
# iris_df.columns() -> ¡ERROR! columns es un atributo, no un método.
iris_df.columns


In [ ]:
iris_df.index


In [ ]:
# Cambiar el nombre de las columnas: reemplazar los "." por "_"
iris_df.columns = iris_df.columns.str.replace(".", "_", regex=False)
iris_df.head()


### Indexación y slicing en DataFrames

- `df["columna"]` o `df.columna` → selecciona **una columna** (devuelve una `Series`).
- `df[["col1", "col2"]]` → selecciona **varias columnas** (devuelve un `DataFrame`).
- `df.loc[etiquetas]` → selección **por etiqueta** de índice.
- `df.iloc[posiciones]` → selección **por posición** entera (como en NumPy).


In [ ]:
# Selección de una columna (-> Series) y de varias columnas (-> DataFrame)
print(type(iris_df["sepal_length"]))
print(type(iris_df[["sepal_length", "petal_length"]]))

iris_df[["sepal_length", "petal_length"]].head()


In [ ]:
# Slicing por posición e indexación booleana
iris_df.iloc[0:3]        # primeras 3 filas


In [ ]:
iris_df.loc[iris_df["variety"] == "Virginica"]   # filtro booleano, equivalente a un WHERE


### Cambiar índice y nombres de columnas

- `DataFrame.index = nueva_lista` / `DataFrame.columns = nueva_lista` → reemplaza **todo** el índice o todas las columnas.
- `df.rename(index={...}, columns={...})` → reemplaza **solo algunas** etiquetas. Por defecto **no modifica el original**, salvo que uses `inplace=True`.
- `df.reset_index()` → convierte el índice actual en una columna y pone un índice numérico por defecto.
- `df.set_index("columna")` → usa una columna existente como nuevo índice.


In [ ]:
alumnos = pd.DataFrame({
    "student_name": ["Ana", "Bruno", "Caro", "Denis"],
    "kor": [90, 85, 78, 92],
})
alumnos.index = ["stu1", "stu2", "stu3", "stu4"]
alumnos


In [ ]:
# rename() sin inplace -> devuelve una copia, el original queda igual
alumnos.rename(index={"stu2": "estudiante2"})


In [ ]:
print("¿Cambió el original?")
print(alumnos.index.tolist())   # sigue igual

# Con inplace=True sí se modifica el objeto original
alumnos.rename(index={"stu2": "estudiante2"}, inplace=True)
alumnos.rename(columns={"student_name": "stu_name"}, inplace=True)
alumnos


In [ ]:
# Agregar una columna nueva y luego usar una columna existente como índice
alumnos["nombre_col_extra"] = "prueba"
alumnos = alumnos.set_index("stu_name")
alumnos


In [ ]:
# reset_index() devuelve el índice a una columna normal y pone un índice numérico 0..n-1
alumnos.reset_index()


### Eliminar filas y columnas: `drop()`

- Para eliminar una **fila**: `df.drop(etiqueta_o_lista)` o `axis=0` (por defecto).
- Para eliminar una **columna**: `df.drop(nombre_o_lista, axis=1)`.
- **Buena práctica**: copiá el DataFrame original (`df.copy()`) antes de eliminar, así podés recuperar los datos si te equivocás.


In [ ]:
df_original = iris_df.copy()

# Eliminar una columna
df_sin_petalo = df_original.drop("petal_width", axis=1)
print(df_sin_petalo.columns.tolist())

# Eliminar varias filas por índice
df_sin_filas = df_original.drop([0, 1], axis=0)
df_sin_filas.head()


### ✏️ Ejercicio 2.2

1. Creá un `DataFrame` llamado `productos` con las columnas `nombre`, `precio` y `stock`, con al menos 4 productos inventados.
2. Agregá una nueva columna llamada `precio_con_iva` que sea `precio * 1.21`.
3. Renombrá la columna `stock` a `unidades_disponibles` **sin** modificar el DataFrame original (o sea, sin `inplace=True`).
4. Mostrá solo los productos cuyo `precio` sea mayor a 1000.


In [ ]:
# 👉 Tu solución acá



<details>
<summary>💡 Ver solución</summary>

```python
productos = pd.DataFrame({
    "nombre": ["Mouse", "Teclado", "Monitor", "Notebook"],
    "precio": [3500, 8200, 95000, 650000],
    "stock":  [50, 30, 12, 5],
})

productos["precio_con_iva"] = productos["precio"] * 1.21

productos_renombrado = productos.rename(columns={"stock": "unidades_disponibles"})

productos[productos["precio"] > 1000]
```
</details>


## 2.3. Fusionar y vincular DataFrames

Cuando tenemos información repartida en varios DataFrames, pandas ofrece dos formas principales de combinarlos:

- **`pd.concat()`**: "pega" DataFrames uno debajo (o al lado) de otro, según el eje (`axis=0` filas, `axis=1` columnas). Es una unión simple, no requiere una columna en común.
- **`pd.merge()`**: fusiona dos DataFrames **según una clave común** (columna o índice), de forma similar a un `JOIN` de SQL. La clave debe existir en ambos DataFrames.


In [ ]:
df1 = pd.DataFrame({"a": [1, 2], "b": [3, 4], "c": [5, 6]})
df2 = pd.DataFrame({"a": [7, 8], "b": [9, 10], "d": [11, 12]})

print("df1:")
display(df1)
print("df2:")
display(df2)


In [ ]:
# concat por filas (axis=0, por defecto): si faltan columnas, se completan con NaN
pd.concat([df1, df2], axis=0)


In [ ]:
# reset del índice para que no queden duplicados
pd.concat([df1, df2], axis=0, ignore_index=True)


In [ ]:
# concat por columnas: pega los DataFrames "lado a lado"
pd.concat([df1, df2], axis=1)


In [ ]:
# merge(): fusión basada en una clave común, similar a un JOIN de SQL
clientes = pd.DataFrame({
    "id_cliente": [1, 2, 3, 4],
    "nombre":     ["Ana", "Bruno", "Caro", "Denis"],
})

pedidos = pd.DataFrame({
    "id_cliente": [1, 2, 2, 5],
    "producto":   ["Mouse", "Teclado", "Monitor", "Notebook"],
})

# merge "inner" (por defecto): solo las claves que están en AMBOS DataFrames
pd.merge(clientes, pedidos, on="id_cliente")


In [ ]:
# Otros tipos de join: left, right, outer
print("LEFT (todos los clientes, aunque no tengan pedido):")
display(pd.merge(clientes, pedidos, on="id_cliente", how="left"))

print("OUTER (todo lo que aparezca en cualquiera de los dos):")
display(pd.merge(clientes, pedidos, on="id_cliente", how="outer"))


> Si `df1` y `df2` **no tienen ningún nombre de columna en común**, `pd.merge()` da error — hay que indicar explícitamente las claves con `left_on=` y `right_on=`.


### ✏️ Ejercicio 2.3

1. Creá dos DataFrames: `empleados` (columnas `id_emp`, `nombre`, `id_depto`) y `departamentos` (columnas `id_depto`, `nombre_depto`).
2. Usá `pd.merge()` para obtener una tabla con el nombre del empleado y el nombre de su departamento.
3. ¿Qué pasa si un empleado tiene un `id_depto` que no existe en `departamentos`? Probá con `how="left"` y con `how="inner"` y compará los resultados.


In [ ]:
# 👉 Tu solución acá



<details>
<summary>💡 Ver solución</summary>

```python
empleados = pd.DataFrame({
    "id_emp": [1, 2, 3, 4],
    "nombre": ["Lucía", "Marcos", "Nadia", "Oscar"],
    "id_depto": [10, 20, 20, 99],   # 99 no existe en departamentos
})

departamentos = pd.DataFrame({
    "id_depto": [10, 20, 30],
    "nombre_depto": ["Ventas", "Sistemas", "RRHH"],
})

# inner: Oscar (id_depto=99) desaparece porque no hay match
display(pd.merge(empleados, departamentos, on="id_depto", how="inner"))

# left: Oscar se mantiene, con NaN en nombre_depto
display(pd.merge(empleados, departamentos, on="id_depto", how="left"))
```
</details>


## 2.4. Clasificación de DataFrames e índice múltiple

### Ordenar (`sort_values`, `sort_index`)

- `df.sort_values("columna")` → ordena las filas según los valores de una columna (ascendente por defecto).
- `df.sort_values("columna", ascending=False)` → orden descendente.
- `df.sort_values(["col1", "col2"])` → ordena usando **más de una columna** como criterio.
- `df.sort_index()` → ordena por el índice en lugar de por una columna.


In [ ]:
ventas = pd.DataFrame({
    "vendedor": ["Ana", "Bruno", "Caro", "Denis", "Ana"],
    "region":   ["Norte", "Sur", "Norte", "Sur", "Sur"],
    "monto":    [1500, 3200, 800, 3200, 2100],
})

# Orden ascendente por monto
ventas.sort_values("monto")


In [ ]:
# Orden descendente
ventas.sort_values("monto", ascending=False)


In [ ]:
# Ordenar usando dos columnas: primero por región, y dentro de cada región por monto descendente
ventas.sort_values(["region", "monto"], ascending=[True, False])


### Indexación jerárquica con `MultiIndex`

Un `MultiIndex` permite tener **más de un nivel de índice** (por ejemplo filas o columnas organizadas en "capas"), útil para representar datos con más de una dimensión categórica.


In [ ]:
# Construimos un MultiIndex de columnas: capa exterior = grupo, capa interior = variable
capa_exterior = ["Grupo A", "Grupo A", "Grupo B", "Grupo B"]
capa_interior = ["x", "y", "x", "y"]

tuplas = list(zip(capa_exterior, capa_interior))
columnas_multi = pd.MultiIndex.from_tuples(tuplas, names=["grupo", "variable"])

datos_multi = pd.DataFrame(
    np.round(np.random.default_rng(42).normal(size=(4, 4)), 2),
    columns=columnas_multi,
)
datos_multi


In [ ]:
# Acceder a un "grupo" completo (capa exterior)
datos_multi["Grupo A"]


In [ ]:
# MultiIndex también se puede usar en las FILAS
indice_filas = pd.MultiIndex.from_product(
    [["2024", "2025"], ["Q1", "Q2"]], names=["anio", "trimestre"]
)

ventas_trimestrales = pd.DataFrame(
    {"monto": [12000, 15000, 13500, 17800]}, index=indice_filas
)
ventas_trimestrales


In [ ]:
# Con MultiIndex en filas, .loc permite "entrar" nivel por nivel
ventas_trimestrales.loc["2024"]


### ✏️ Ejercicio 2.4 — `ex_0108`

Siguiendo el espíritu del ejercicio de la clase (`ex_0108.ipynb`):

1. A partir del DataFrame `ventas` de arriba, ordená las filas por `vendedor` (alfabético ascendente) y, dentro de cada vendedor, por `monto` descendente.
2. Armá un `MultiIndex` de filas con niveles `region` y `vendedor`, y usalo como índice de una copia de `ventas`.
3. Usando ese índice múltiple, seleccioná todas las filas de la región `"Sur"`.


In [ ]:
# 👉 Tu solución acá



<details>
<summary>💡 Ver solución</summary>

```python
# 1)
ventas_ordenadas = ventas.sort_values(["vendedor", "monto"], ascending=[True, False])
display(ventas_ordenadas)

# 2)
ventas_multi = ventas.set_index(["region", "vendedor"]).sort_index()
display(ventas_multi)

# 3)
display(ventas_multi.loc["Sur"])
```
</details>


## 2.5. Estadísticas descriptivas, agrupamiento y muestreo de datos

### Agrupar y resumir: `groupby()`

`groupby()` agrupa las filas según los valores de una o más columnas, para después aplicarles una función de agregación (`sum`, `mean`, `count`, `size`, etc.) — el equivalente a `GROUP BY` de SQL.


In [ ]:
ventas.groupby("region")["monto"].sum()


In [ ]:
# Se puede agrupar por más de una columna y aplicar varias agregaciones a la vez
ventas.groupby(["region", "vendedor"])["monto"].agg(["sum", "mean", "count"])


### Pivotando: `pivot_table()`

`pivot_table()` reorganiza los datos: transforma valores únicos de una columna en **nuevas columnas**, usando otra columna como índice y una tercera como valores a agregar.


In [ ]:
# index = filas, columns = nuevas columnas, values = qué agregamos, aggfunc = cómo agregamos
tabla_pivot = ventas.pivot_table(
    index="vendedor", columns="region", values="monto", aggfunc="sum"
)
tabla_pivot


In [ ]:
# Igual que arriba, pero completando los valores faltantes con 0 en lugar de NaN
ventas.pivot_table(
    index="vendedor", columns="region", values="monto", aggfunc="sum", fill_value=0
)


### Estadísticas descriptivas

- `df.describe()` → resumen estadístico (count, mean, std, min, cuartiles, max) de las columnas numéricas.
- `df["columna"].value_counts()` → cuenta cuántas veces aparece cada valor único (útil para columnas categóricas).
- `df.corr(numeric_only=True)` → matriz de correlación entre columnas numéricas.


In [ ]:
iris_df.describe()


In [ ]:
iris_df["variety"].value_counts()


In [ ]:
iris_df.corr(numeric_only=True)


### Muestreo de datos: `sample()`

`df.sample(n)` devuelve `n` filas elegidas **al azar** — muy útil para inspeccionar rápidamente un dataset grande sin mirar siempre las mismas primeras filas.


In [ ]:
iris_df.sample(3, random_state=42)


In [ ]:
# frac= devuelve una FRACCIÓN del total de filas en lugar de una cantidad fija
iris_df.sample(frac=0.3, random_state=42)


### Convertir variables continuas en categóricas: `pd.cut()`

`pd.cut()` divide una variable numérica continua en **contenedores (bins)** discretos — muy usado en *feature engineering* para convertir, por ejemplo, edades en rangos etarios.


In [ ]:
# Ejemplo: convertir la longitud del pétalo (continua) en 3 categorías: chico / mediano / grande
valores = iris_df["petal_length"]

# Límites de los bins (podrían salir de np.histogram, o definirse a mano como acá)
limites = [0, 2, 5, 7]
nombres_bins = ["chico", "mediano", "grande"]

iris_df["petal_length_bin"] = pd.cut(
    valores, bins=limites, labels=nombres_bins, include_lowest=True
)

iris_df[["petal_length", "petal_length_bin"]].head(10)


In [ ]:
iris_df["petal_length_bin"].value_counts()


### Variables ficticias (*dummy variables*): `pd.get_dummies()`

Para usar datos categóricos en algoritmos de Machine Learning, muchas veces hay que convertirlos en variables numéricas 0/1 — una columna por cada categoría.


In [ ]:
pd.get_dummies(iris_df["petal_length_bin"], prefix="petalo")


### ✏️ Ejercicio 2.5

1. Con `iris_df`, agrupá por `variety` y calculá el promedio (`mean`) de `sepal_length` y `petal_length` para cada variedad.
2. Armá una `pivot_table` con `variety` como índice y el promedio de `petal_width` como valor.
3. Creá una nueva columna `sepal_length_bin` que divida `sepal_length` en 2 categorías (`"chica"`, `"grande"`) usando `pd.cut()`.
4. Tomá una muestra aleatoria de 4 filas de `iris_df` con `random_state=7`.


In [ ]:
# 👉 Tu solución acá



<details>
<summary>💡 Ver solución</summary>

```python
# 1)
display(iris_df.groupby("variety")[["sepal_length", "petal_length"]].mean())

# 2)
display(iris_df.pivot_table(index="variety", values="petal_width", aggfunc="mean"))

# 3)
iris_df["sepal_length_bin"] = pd.cut(
    iris_df["sepal_length"], bins=2, labels=["chica", "grande"]
)
display(iris_df[["sepal_length", "sepal_length_bin"]])

# 4)
display(iris_df.sample(4, random_state=7))
```
</details>


## 🎯 Resumen de la unidad

| Tema | Función / atributo clave |
|---|---|
| Crear estructuras | `pd.Series()`, `pd.DataFrame()` |
| Inspeccionar | `.head()`, `.tail()`, `.info()`, `.describe()`, `.shape`, `.columns`, `.index` |
| Seleccionar | `df["col"]`, `df[["c1","c2"]]`, `.loc[]`, `.iloc[]` |
| Modificar estructura | `.rename()`, `.reset_index()`, `.set_index()`, `.drop()` |
| Combinar DataFrames | `pd.concat()`, `pd.merge()` |
| Ordenar | `.sort_values()`, `.sort_index()`, `pd.MultiIndex` |
| Resumir | `.groupby()`, `.pivot_table()`, `.corr()`, `.value_counts()` |
| Muestrear | `.sample()` |
| Discretizar | `pd.cut()`, `pd.get_dummies()` |

¡Con esto ya tenés las herramientas centrales de `pandas` que se usan como base para el resto del curso (limpieza de datos, feature engineering y análisis exploratorio)! 🚀
